# All Model saves here
 Option 1: Split by scene (first 75% of scenes for training, last 25% for validation)
- solve mpos part and transformer
- transformer layer 6
- trian / val 2 : 1
- 0~14 train / 15 predict
- 

## Question
- this train/ val 1 : 1 -
- train : past observation training : scene[0:14] target: scene[14]
  val : past observation training : scene[15:29] target: scene[29]
- Then it will train one train
- but option 2 and option 3 which is user split and subcarrier split
- train : past obervation training : scene[0:14] ~ scene[15:29] which doesn't overlap user and subcarrier
- val : same train not overlap user or subcarrier
- so Option 1 can train 1 time but option 2, 3 train 16 times

## import

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader, random_split
import DeepMIMOv3
import numpy as np
from pprint import pprint
import matplotlib.pyplot as plt
import time
import math
import torch
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import IterableDataset
import numpy as np
import time, gc
from tqdm import tqdm
import numpy as np
import torch
import random
import torch.nn as nn
from lwm_model import lwm
from torch.optim import Adam
from pathlib import Path
import torch, time



In [2]:
start = time.time()

## GPU Settings

In [3]:
# GPU 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
import torch
print(torch.version.cuda)                   
print(torch.backends.cudnn.version())       
print("CUDA available:", torch.cuda.is_available())  # True

12.6
90501
CUDA available: True


## DeepMIMOv3 dataset

In [5]:
parameters = DeepMIMOv3.default_params()

In [6]:
## Change parameters for the setup
# Scenario O1_60 extracted at the dataset_folder
#LWM dynamic senario
# parameters['dataset_folder'] = r'/content/drive/MyDrive/Colab Notebooks/LWM'
scene = 44# scene 60
# change my linux route
parameters['dataset_folder'] = '/home/dlghdbs200/LWM/scenarios'

# scnario = 02_dyn_3p5 <- download file
parameters['scenario'] = 'O2_dyn_3p5'
parameters['dynamic_scenario_scenes'] = np.arange(scene) #scene 0~9

# Up to 10 multipath paths per user-to-base station channel
parameters['num_paths'] = 10

# User rows 1-100
parameters['user_rows'] = np.arange(100)
# User subsampling
parameters['user_subsampling'] = 0.01

# Activate only the first basestation
parameters['active_BS'] = np.array([1])

parameters['activate_OFDM'] = 1

parameters['OFDM']['bandwidth'] = 0.05 # 50 MHz
parameters['OFDM']['subcarriers'] = 512 # OFDM with 512 subcarriers
parameters['OFDM']['selected_subcarriers'] = np.arange(0, 64, 1)
#parameters['OFDM']['subcarriers_limit'] = 64 # Keep only first 64 subcarriers

parameters['ue_antenna']['shape'] = np.array([1, 1]) # Single antenna
parameters['bs_antenna']['shape'] = np.array([1, 32]) # ULA of 32 elements
#parameters['bs_antenna']['rotation'] = np.array([0, 30, 90]) # ULA of 32 elements
#parameters['ue_antenna']['rotation'] = np.array([[0, 30], [30, 60], [60, 90]]) # ULA of 32 elements
#parameters['ue_antenna']['radiation_pattern'] = 'isotropic'
#parameters['bs_antenna']['radiation_pattern'] = 'halfwave-dipole'

In [7]:
## dataset setting (chunked on‑the‑fly generation)
import time, gc
from tqdm import tqdm

# 0~999 scene index , process 50 at that time
scene_indices = np.arange(scene)
chunk_size   = 5
all_data     = []

# Call generate_data for each scene chunk
for i in tqdm(range(0, len(scene_indices), chunk_size)):
    chunk = scene_indices[i : i+chunk_size].tolist()
    parameters['dynamic_scenario_scenes'] = chunk

    start = time.time()
    data_chunk = DeepMIMOv3.generate_data(parameters)
    print(f"Scenes {chunk[0]}–{chunk[-1]} generation time: {time.time() - start:.2f}s")

    # combine all_data or save in the Disk
    all_data.extend(data_chunk)

    # free memory 
    del data_chunk
    gc.collect()

# comvine Dataset
dataset = all_data


print(parameters['user_rows'])

  0%|                                                                                             | 0/9 [00:00<?, ?it/s]

The following parameters seem unnecessary:
{'activate_OFDM'}

Scene 1/5

Basestation 1

UE-BS Channels



Reading ray-tracing: 100%|█████████████████████████████████████████████████████| 69006/69006 [00:00<00:00, 88839.28it/s]

Generating channels: 100%|██████████████████████████████████████████████████████████| 727/727 [00:00<00:00, 7522.81it/s]



BS-BS Channels



Reading ray-tracing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 4750.06it/s]

Generating channels: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 624.34it/s]


...


Scene 4/4

Basestation 1

UE-BS Channels



Reading ray-tracing: 100%|████████████████████████████████████████████████████| 69006/69006 [00:00<00:00, 286366.30it/s]

Generating channels: 100%|██████████████████████████████████████████████████████████| 727/727 [00:00<00:00, 7390.21it/s]



BS-BS Channels



Reading ray-tracing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6269.51it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████| 9/9 [01:03<00:00,  7.01s/it]

Scenes 40–43 generation time: 5.75s
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]


## About Information
User : 737
UE antenna : 1
BS antenna : 32  Shape(a+bj)
subcarrier : 64

In [8]:
# Unmasked Data Model(gru
# separate maksed data and unmasked data

In [9]:
len(dataset)

44

## Data Preprocessing

In [10]:
import numpy as np
import torch
from torch.utils.data import IterableDataset
from sklearn.preprocessing import MinMaxScaler

def concat_channel(h: np.ndarray) -> np.ndarray:
    """
    Convert a complex channel vector to a real-valued vector by
    concatenating real and imaginary parts.
    """
    return np.concatenate([h.real, h.imag]).astype(np.float32)

class UnMaskedChannelSeqDataset(IterableDataset):
    """
    Iterable dataset for next-step channel vector prediction.

    - Task: Given seq_len past channel observations, predict the next channel vector.
    - Data processing:
        1. Extract and flatten real + imaginary parts for each channel vector.
        2. Apply Min-Max scaling (shared scaler for inputs and another for targets).
        3. Iterate over all time steps, users, and subcarriers to yield (sequence, target).
    """
    def __init__(
        self,
        scenes,
        seq_len: int = 5,
        eps: float = 1e-9,
        scalers: tuple[MinMaxScaler, MinMaxScaler] | None = None,
    ):
        super().__init__()
        self.scenes = scenes        # List of scenes, each containing channel data
        self.seq_len = seq_len      # Number of past steps to use for prediction
        self.eps = eps              # Small epsilon (unused here)

        # Determine dataset dimensions from the first scene
        ch0 = scenes[0][0]['user']['channel']
        self.U = ch0.shape[0]       # Number of users
        self.A = ch0.shape[2]       # Number of antennas
        self.S = ch0.shape[3]       # Number of subcarriers
        self.vec_len = 2 * self.A   # Length of concatenated vector (real+imag)

        # Initialize scalers if not provided
        if scalers is None:
            self.scaler_x = MinMaxScaler()
            self.scaler_y = MinMaxScaler()
            T = len(scenes)

            # Fit scalers over all valid sequences and targets
            for t in range(self.seq_len, T):
                past = scenes[t - self.seq_len:t]
                tgt_scene = scenes[t]

                for u in range(self.U):
                    for s in range(self.S):
                        # Build sequence of flattened vectors
                        seq_np = np.stack([
                            concat_channel(p[0]['user']['channel'][u, 0, :, s])
                            for p in past
                        ], axis=0)
                        tgt_np = concat_channel(
                            tgt_scene[0]['user']['channel'][u, 0, :, s]
                        )

                        # Skip if sequence or target is all zeros
                        if not np.any(seq_np) or not np.any(tgt_np):
                            continue

                        # Fit scalers incrementally
                        self.scaler_x.partial_fit(seq_np.reshape(-1, self.vec_len))
                        self.scaler_y.partial_fit(tgt_np.reshape(1, -1))
        else:
            self.scaler_x, self.scaler_y = scalers

    def __iter__(self):
        """
        Yield scaled (sequence, target) pairs as torch.FloatTensor.
        """
        T = len(self.scenes)
        for t in range(self.seq_len, T):
            past = self.scenes[t - self.seq_len:t]
            tgt_scene = self.scenes[t]

            for u in range(self.U):
                for s in range(self.S):
                    seq_np = np.stack([
                        concat_channel(p[0]['user']['channel'][u, 0, :, s])
                        for p in past
                    ], axis=0)
                    tgt_np = concat_channel(
                        tgt_scene[0]['user']['channel'][u, 0, :, s]
                    )

                    # Skip empty data
                    if not np.any(seq_np) or not np.any(tgt_np):
                        continue

                    # Apply fitted scalers
                    N, D = seq_np.shape
                    seq_scaled = self.scaler_x.transform(seq_np.reshape(-1, D)).reshape(N, D)
                    tgt_scaled = self.scaler_y.transform(tgt_np.reshape(1, -1)).reshape(-1,)

                    yield torch.from_numpy(seq_scaled), torch.from_numpy(tgt_scaled)

    def __len__(self):
        """
        Total number of valid (sequence, target) samples in the dataset.
        """
        return (len(self.scenes) - self.seq_len) * self.U * self.S


In [11]:
!nvidia-smi


Tue Aug 12 17:01:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.02              Driver Version: 576.02         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0  On |                  N/A |
| N/A   48C    P5             17W /  140W |     649MiB /   6144MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 

In [12]:
import numpy as np
import torch
import random
from torch.utils.data import IterableDataset
from sklearn.preprocessing import MinMaxScaler

def concat_channel(h: np.ndarray) -> np.ndarray:
    """
    Convert a complex channel vector to a real-valued vector by
    concatenating its real and imaginary parts.
    """
    return np.concatenate([h.real, h.imag]).astype(np.float32)

class MaskedChannelSeqDataset(IterableDataset):
    """
    Iterable dataset for next-step channel vector prediction with optional masking.

    - Task: Given seq_len past channel observations, predict the next channel vector.
    - Data processing:
      1. Flatten each complex channel patch into a real vector (real+imag).
      2. Apply Min-Max scaling to inputs and targets (shared per-sequence and per-target).
      3. Randomly mask one time-step in the sequence (15% probability):
         * 80% zero-out mask
         * 10% Gaussian noise mask
         * 10% keep original value
    - Outputs:
      masked_sequence (torch.FloatTensor of shape [seq_len, vec_len]),
      mask_position (torch.LongTensor of shape [1]),
      target_vector (torch.FloatTensor of shape [vec_len]).
    - Supports reuse of external scalers for consistent train/validation splits.
    """
    def __init__(
        self,
        scenes,
        seq_len: int = 5,
        eps: float = 1e-9,
        noise_std: float = 1.0,
        scalers: tuple[MinMaxScaler, MinMaxScaler] | None = None
    ):
        super().__init__()
        self.scenes = scenes        # List of scene dicts containing channel data
        self.seq_len = seq_len      # Number of past steps to use
        self.eps = eps              # Epsilon (currently unused)
        self.noise_std = noise_std  # Standard deviation for noise masking

        # Infer dimensions from the first scene
        ch0 = scenes[0][0]['user']['channel']  # (U, 1, A, S)
        self.U, _, self.A, self.S = ch0.shape
        self.vec_len = 2 * self.A               # Length of flattened vector

        # Initialize or reuse scalers
        if scalers is None:
            self.scaler_x = MinMaxScaler()
            self.scaler_y = MinMaxScaler()
            self._fit_scalers()
        else:
            self.scaler_x, self.scaler_y = scalers

        # Predefine zero-vector for zero-mask
        self.mask_value = torch.zeros(self.vec_len, dtype=torch.float32)

    def _fit_scalers(self):
        """
        Incrementally fit MinMax scalers over all valid sequences and targets.
        """
        T = len(self.scenes)
        for t in range(self.seq_len, T):
            past = self.scenes[t - self.seq_len : t]
            tgt_scene = self.scenes[t]
            for u in range(self.U):
                for s in range(self.S):
                    seq_np = np.stack([
                        concat_channel(p[0]['user']['channel'][u, 0, :, s])
                        for p in past
                    ], axis=0)
                    tgt_np = concat_channel(
                        tgt_scene[0]['user']['channel'][u, 0, :, s]
                    )
                    if not np.any(seq_np) or not np.any(tgt_np):
                        continue
                    self.scaler_x.partial_fit(seq_np.reshape(-1, self.vec_len))
                    self.scaler_y.partial_fit(tgt_np.reshape(1, -1))

    def __iter__(self):
        """
        Yield tuples: (masked_sequence, mask_position, target_vector).
        Mask one patch per sequence with 15% chance.
        """
        mask_prob  = 0
        zero_prob  = mask_prob * 0.8
        noise_prob = mask_prob * 0.1

        T = len(self.scenes)
        for t in range(self.seq_len, T):
            past = self.scenes[t - self.seq_len : t]
            tgt_scene = self.scenes[t]

            for u in range(self.U):
                for s in range(self.S):
                    seq_np = np.stack([
                        concat_channel(p[0]['user']['channel'][u, 0, :, s])
                        for p in past
                    ], axis=0)
                    tgt_np = concat_channel(
                        tgt_scene[0]['user']['channel'][u, 0, :, s]
                    )
                    if not np.any(seq_np) or not np.any(tgt_np):
                        continue

                    # Scale sequences and targets
                    N, D = seq_np.shape
                    seq_scaled = self.scaler_x.transform(seq_np.reshape(-1, D)).reshape(N, D)
                    tgt_scaled = self.scaler_y.transform(tgt_np.reshape(1, -1)).reshape(-1,)

                    seq_tensor = torch.from_numpy(seq_scaled)
                    tgt_tensor = torch.from_numpy(tgt_scaled)

                    # Select position to mask
                    mpos = random.randrange(self.seq_len)
                    r = random.random()

                    if r < zero_prob:
                        masked_seq = seq_tensor.clone()
                        masked_seq[mpos] = self.mask_value
                    elif r < zero_prob + noise_prob:
                        masked_seq = seq_tensor.clone()
                        masked_seq[mpos] = torch.randn(self.vec_len) * self.noise_std
                    elif r < mask_prob:
                        masked_seq = seq_tensor  # mask index but leave value
                    else:
                        masked_seq = seq_tensor  # no masking

                    yield masked_seq, torch.tensor([mpos]), tgt_tensor

    def __len__(self):
        """
        Total number of valid (sequence, position, target) samples.
        """
        return (len(self.scenes) - self.seq_len) * self.U * self.S


## Split Train/Val

In [13]:
# seq_len = 14 -> past 14 target 1

seq_len      = 14
batch_size = 256

split_idx    = 26

train_ds = dataset[:split_idx]
val_ds = dataset[split_idx:]

In [14]:
import psutil

mem = psutil.virtual_memory()
print(f"Used: {mem.used / 1024**2:.2f} MB")
print(f"Available: {mem.available / 1024**2:.2f} MB")
print(f"Total: {mem.total / 1024**2:.2f} MB")


Used: 4113.29 MB
Available: 3363.18 MB
Total: 7566.18 MB


In [15]:
type(train_ds[0][0]['user']['channel'][2])

numpy.ndarray

In [16]:
print("type s_tgt:", type(train_ds[0]))
print("type s_tgt[0]:", type(train_ds[0][0]))


type s_tgt: <class 'list'>
type s_tgt[0]: <class 'dict'>


In [17]:
type(train_ds[0][0])

dict

In [18]:
train_ds[0][0]['user']['channel'][3]

array([[[-7.7230516e-06-3.7325199e-06j, -7.9651563e-06-3.0329672e-06j,
         -8.1445578e-06-2.3225596e-06j, ...,
          4.0902052e-07-9.2439504e-06j, -4.2611231e-07-9.2992277e-06j,
         -1.2701220e-06-9.2776154e-06j],
        [-8.9658297e-06-5.0922256e-07j, -8.9196074e-06+2.7402834e-07j,
         -8.8062361e-06+1.0425907e-06j, ...,
         -3.1661684e-06-8.7219523e-06j, -3.9539450e-06-8.4581316e-06j,
         -4.7217295e-06-8.1212092e-06j],
        [-8.8458492e-06+3.0791930e-06j, -8.4874573e-06+3.8251083e-06j,
         -8.0681375e-06+4.5295747e-06j, ...,
         -6.2886138e-06-6.8233676e-06j, -6.9076514e-06-6.2828899e-06j,
         -7.4816817e-06-5.6838990e-06j],
        ...,
        [ 1.7968513e-06-4.9266268e-06j,  1.4072244e-06-5.0916565e-06j,
          1.0005344e-06-5.2284004e-06j, ...,
          6.3669818e-06-3.4926693e-06j,  5.9878876e-06-4.0733821e-06j,
          5.5549508e-06-4.6118416e-06j],
        [-1.2760374e-07-4.8192383e-06j, -5.0755955e-07-4.8266352e-06j,
    

In [19]:
train_ds[0][0]['user']['channel'][2]

array([[[-1.3557186e-06+6.6490680e-09j, -1.3003229e-06+3.8363723e-07j,
         -1.1422120e-06+7.3032106e-07j, ...,
          9.8271038e-08-1.3521685e-06j, -2.8187318e-07-1.3261088e-06j,
         -6.3975170e-07-1.1952969e-06j],
        [-1.3461549e-06+1.6088443e-07j, -1.2482185e-06+5.2911969e-07j,
         -1.0516826e-06+8.5555865e-07j, ...,
         -5.6241912e-08-1.3545678e-06j, -4.3095110e-07-1.2854175e-06j,
         -7.7161860e-07-1.1147295e-06j],
        [-1.3191018e-06+3.1302952e-07j, -1.1798969e-06+6.6772765e-07j,
         -9.4748964e-07+9.6968063e-07j, ...,
         -2.1002415e-07-1.3393681e-06j, -5.7443003e-07-1.2280258e-06j,
         -8.9346048e-07-1.0196791e-06j],
        ...,
        [ 1.3360471e-06-2.3020671e-07j,  1.2192198e-06-5.9289130e-07j,
          1.0060838e-06-9.0874221e-07j, ...,
          1.2613847e-07+1.3498541e-06j,  4.9677539e-07+1.2614402e-06j,
          8.2817098e-07+1.0733825e-06j],
        [ 1.3011709e-06-3.8075123e-07j,  1.1438295e-06-7.2778505e-07j,
    

# DataLoader
Samples = (len(self.scenes) - self.seq_len) * self.U * self.S / 32

In [20]:

unmasked_train_ds = UnMaskedChannelSeqDataset(train_ds, seq_len=seq_len)
unmasked_val_ds   = UnMaskedChannelSeqDataset(val_ds, seq_len=seq_len)

# iterate over train_ds to compute min and max of features/targets

unmasked_train_loader = DataLoader(unmasked_train_ds, batch_size=batch_size, shuffle=False)
unmasked_val_loader   = DataLoader(unmasked_val_ds,   batch_size=batch_size, shuffle=False)
# ─────────────────────────────────────────────


In [21]:
# ❷ Train/Validation DataLoader split train : val = 3 : 1

masked_train_ds = MaskedChannelSeqDataset(train_ds, seq_len=seq_len)
masked_val_ds   = MaskedChannelSeqDataset(val_ds, seq_len=seq_len)

# iterate over train_ds to compute min and max of features/targets

masked_train_loader = DataLoader(masked_train_ds, batch_size=batch_size, shuffle=False)
masked_val_loader   = DataLoader(masked_val_ds,   batch_size=batch_size, shuffle=False)
# ─────────────────────────────────────────────


1454

In [22]:
len(unmasked_train_ds)

558336

In [23]:
len(masked_train_loader)

2181

In [24]:
len(masked_val_loader)

727

## Define Model

LWMWithHead: A wrapper class that uses a pre-trained LWM (Transformer encoder) as the backbone,
             and attaches a new fully-connected (FC) head for downstream tasks
             (regression, classification, etc.).

Changes:
- input_dim: Dimension of the actual input data (e.g., 64)
- patch_length: Patch length expected by the backbone (e.g., 16)
- Replaces the original element_length parameter with these two distinct parameters
- Applies a projection layer (self.input_proj) in forward()


In [25]:
class LWMWithHead(nn.Module):
    """
    LWMWithHead: A wrapper class that uses a pre-trained LWM (Transformer encoder) as the backbone,
                 and attaches a new fully-connected (FC) head for downstream tasks
                 (regression, classification, etc.).

    Changes:
    - input_dim: Dimension of the actual input data (e.g., 64)
    - patch_length: Patch length expected by the backbone (e.g., 16)
    - Replaces the original element_length parameter with these two distinct parameters
    - Applies a projection layer (self.input_proj) in forward()
    """
    def __init__(
        self,
        patch_length: int = 64,         # Patch length expected by the backbone (e.g., 64)
        d_model: int = 64,              # LWM hidden size
        max_len: int = 129,             # Positional encoding max length
        n_layers: int = 12,             # Number of Transformer encoder layers
        out_dim: int = 64,              # FC head output dimension
        freeze_backbone: bool = True,   # Whether to freeze the backbone
        checkpoint_path: str | None = "./model_weights.pth",
        device: str = "cuda"
    ):
        super().__init__()

        # apply a projection layer to match backbone's expected patch_length

        # initialize backbone
        if checkpoint_path is None:
            # randomly initialized backbone
            self.backbone = lwm(
                element_length=patch_length,
                d_model=d_model,
                max_len=max_len,
                n_layers=n_layers
            ).to(device)
        else:
            # load pre-trained weights
            self.backbone = lwm.from_pretrained(
                ckpt_name=checkpoint_path,
                device=device,
                element_length=patch_length,
                d_model=d_model,
                max_len=max_len,
                n_layers=n_layers
            )


        # freeze backbone parameters if required
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # attach a new fully-connected head for downstream tasks
        self.head = nn.Sequential(
            # change 2 layer -> 1 layer
            nn.Linear(d_model, out_dim),
        )

    def forward(self, input_ids: torch.Tensor, masked_pos: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: Tensor of shape (B, L, input_dim)
            masked_pos: Tensor of shape (B, num_mask)
        Returns:
            out: Tensor of shape (B, out_dim)
        """
        # input_ids shape -> (Batch_size, seq_len, elemente_length=path_length)
        x = input_ids
        # backbone forward: returns (logits_lm, enc_output)
        _, enc_output = self.backbone(x, masked_pos)

        # extract CLS token feature (first token)
        feat = enc_output[:, 0, :]

        # pass through FC head to get final output
        out = self.head(feat)
        return out


In [26]:
import torch
import torch.nn as nn

class GRUWithHead(nn.Module):
    """
    GRUWithHead (projected):
      • Projects the raw feature dimension (input_dim) to a smaller patch_length
        so every backbone receives the same patch-sized input (like LWM).
      • Stacks N GRU layers, then an FC head for downstream tasks.
    """
    def __init__(
        self,
        patch_length: int = 64,   # target dimension fed to the GRU backbone
        d_model: int      = 64,   # GRU hidden size
        n_layers: int     = 3,   # number of stacked GRU layers
        bidirectional: bool = True,
        dropout: float      = 0.1,
        hidden_dim: int     = 256, # FC-head hidden size
        out_dim: int        = 64,  # FC-head output size
        freeze_backbone: bool = False
    ):
        super().__init__()
        
        # 1) GRU backbone that expects 'patch_length' features per time step
        self.backbone = nn.GRU(
            input_size     = patch_length,
            hidden_size    = d_model,
            num_layers     = n_layers,
            batch_first    = True,
            bidirectional  = bidirectional,
            dropout        = dropout if n_layers > 1 else 0.0
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # 2) Fully-connected head
        gru_out_dim = d_model * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(gru_out_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : Tensor of shape (batch, seq_len, input_dim) – raw features
        Returns:
            Tensor of shape (batch, out_dim)
        """
        # sequence modelling with GRU
        out, _ = self.backbone(x)              # (B, seq_len, num_dirs*d_model)

        # use the last time-step representation
        feat = out[:, -1, :]                        # (B, gru_out_dim)

        # downstream head
        return self.head(feat)                      # (B, out_dim)


In [27]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # Create positional encoding matrix of shape (1, max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        Returns:
            Tensor: x plus positional encodings
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

class InputEmbedding(nn.Module):
    def __init__(self, feat_dim: int, d_model: int, max_len: int = 5000):
        super().__init__()
        # Optional linear projection from feat_dim to d_model
        self.proj = nn.Linear(feat_dim, d_model) if feat_dim != d_model else None
        self.pos_enc = PositionalEncoding(d_model, max_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch, seq_len, feat_dim)
        Returns:
            Tensor of shape (batch, seq_len, d_model)
        """
        if self.proj is not None:
            x = self.proj(x)
        return self.pos_enc(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dim_ff: int, dropout: float = 0.1):
        super().__init__()
        # Multi-Head Self-Attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        # Position-wise Feed-Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model)
        )
        # Layer Normalization and Dropout for residual connections
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        src_mask: torch.Tensor = None,
        src_key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (seq_len, batch, d_model)
            src_mask: Optional Tensor of shape (seq_len, seq_len)
            src_key_padding_mask: Optional Tensor of shape (batch, seq_len)
        Returns:
            Tensor of shape (seq_len, batch, d_model)
        """
        # Self-attention sublayer
        attn_out, _ = self.self_attn(x, x, x, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        x = x + self.dropout1(attn_out)
        x = self.norm1(x)
        # Feed-forward sublayer
        ff_out = self.ff(x)
        x = x + self.dropout2(ff_out)
        x = self.norm2(x)
        return x

class TransformerEncoderCustom(nn.Module):
    def __init__(
        self,
        feat_dim: int,
        d_model: int,
        n_heads: int,
        dim_ff: int,
        n_layers: int,
        dropout: float = 0.1,
        max_len: int = 5000
    ):
        super().__init__()
        # Input embedding: feature projection + positional encoding
        self.input_embedding = InputEmbedding(feat_dim, d_model, max_len)
        # Stack of N encoder layers
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, dim_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(
        self,
        x: torch.Tensor,
        src_mask: torch.Tensor = None,
        src_key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch, seq_len, feat_dim)
        Returns:
            Tensor of shape (seq_len, batch, d_model)
        """
        x = self.input_embedding(x)       # (batch, seq_len, d_model)
        x = x.transpose(0, 1)             # (seq_len, batch, d_model)
        for layer in self.layers:
            x = layer(x, src_mask=src_mask, src_key_padding_mask=src_key_padding_mask)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dim_ff: int, dropout: float = 0.1):
        super().__init__()
        # Masked Self-Attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        # Encoder-Decoder Attention
        self.multihead_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        # Position-wise Feed-Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model)
        )
        # Layer Normalizations and Dropouts
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
        memory_key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            tgt: Tensor of shape (tgt_len, batch, d_model)
            memory: Tensor of shape (src_len, batch, d_model)
        Returns:
            Tensor of shape (tgt_len, batch, d_model)
        """
        # Masked self-attention sublayer
        attn1, _ = self.self_attn(
            tgt, tgt, tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )
        tgt = tgt + self.dropout1(attn1)
        tgt = self.norm1(tgt)
        # Encoder-decoder attention sublayer
        attn2, _ = self.multihead_attn(
            tgt, memory, memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask
        )
        tgt = tgt + self.dropout2(attn2)
        tgt = self.norm2(tgt)
        # Feed-forward sublayer
        ff_out = self.ff(tgt)
        tgt = tgt + self.dropout3(ff_out)
        tgt = self.norm3(tgt)
        return tgt

class TransformerDecoderCustom(nn.Module):
    def __init__(
        self,
        feat_dim: int,
        d_model: int,
        n_heads: int,
        dim_ff: int,
        n_layers: int,
        dropout: float = 0.1,
        max_len: int = 5000
    ):
        super().__init__()
        # Input embedding for target sequence
        self.input_embedding = InputEmbedding(feat_dim, d_model, max_len)
        # Stack of N decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, dim_ff, dropout)
            for _ in range(n_layers)
        ])
        # Final projection back to feature dimension
        # self.output_linear = nn.Linear(d_model, feat_dim)
        self.output_linear = nn.Identity()

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
        memory_key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            tgt: Tensor of shape (batch, tgt_len, feat_dim)
            memory: Tensor of shape (src_len, batch, d_model)
        Returns:
            Tensor of shape (batch, tgt_len, feat_dim)
        """
        x = self.input_embedding(tgt)       # (batch, tgt_len, d_model)
        x = x.transpose(0, 1)               # (tgt_len, batch, d_model)
        for layer in self.layers:
            x = layer(
                x,
                memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask
            )
        x = x.transpose(0, 1)               # (batch, tgt_len, d_model)
        return self.output_linear(x)        # project back to feat_dim

        

class TransformerWithHead(nn.Module):
    def __init__(
        self,
        patch_length: int = 64,   # sequence length consumed by encoder/decoder
        d_model: int      = 64,   # hidden size inside the transformer
        n_heads: int      = 4,
        dim_ff: int       = 256,
        n_layers: int     = 6, # decrease n_layers
        dropout: float    = 0.1,
        out_dim: int      = 64,
        max_len: int      = 5000,
        freeze_backbone: bool = False,
    ):
        super().__init__()



        # 1) Encoder: processes the source sequence
        self.encoder = TransformerEncoderCustom(
            feat_dim = patch_length,
            d_model  = d_model,
            n_heads  = n_heads,
            dim_ff   = dim_ff,
            n_layers = n_layers,
            dropout  = dropout,
            max_len  = max_len,
        )
        if freeze_backbone:
            for p in self.encoder.parameters():
                p.requires_grad = False

        # 2) Decoder: generates target sequence using encoder memory
        self.decoder = TransformerDecoderCustom(
            feat_dim = patch_length,
            d_model  = d_model,
            n_heads  = n_heads,
            dim_ff   = dim_ff,
            n_layers = n_layers,
            dropout  = dropout,
            max_len  = max_len,
        )

        # 3) Task head: maps final decoder output to desired output dimension
        self.head = nn.Sequential(
            nn.Linear(d_model, out_dim)
        )

    def forward(
        self,
        src: torch.Tensor,                # (batch, src_len, input_dim)
        tgt: torch.Tensor,                # (batch, tgt_len, input_dim)
        src_mask: torch.Tensor = None,
        src_key_padding_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None,
        tgt_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        # 1) Encode source sequence to produce memory
        src_patch = src
        memory = self.encoder(
            src_patch,
            src_mask=src_mask,
            src_key_padding_mask=src_key_padding_mask
        )  # (src_len, batch, d_model)

        # 2) Decode target sequence using encoder memory
        tgt_patch = tgt
        dec_out = self.decoder(
            tgt_patch,
            memory,
            tgt_mask=tgt_mask,
            memory_mask=None,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )  # (batch, tgt_len, d_model)

        # 3) Use last time-step output from decoder for prediction
        last_step = dec_out[:, -1, :]      # (batch, d_model)
        return self.head(last_step)        # (batch, out_dim)


In [28]:
class RNNWithHead(nn.Module):
    """
    RNNWithHead (projected):
      • Projects raw feature vectors from `input_dim` to `patch_length`
      • Feeds the projected sequence to an RNN backbone
      • Maps the last hidden state through an FC head
    """
    def __init__(
        self,
        patch_length: int = 64,   # dimension consumed by the RNN backbone
        hidden_size: int  = 64,   # RNN hidden size
        num_layers: int   = 3,   # number of stacked RNN layers
        bidirectional: bool = True,
        dropout: float      = 0.1,
        hidden_dim: int     = 256, # FC-head hidden size
        out_dim: int        = 64,  # FC-head output size
        freeze_backbone: bool = False,
    ):
        super().__init__()
        

        # 1) RNN backbone
        self.backbone = nn.RNN(
            input_size     = patch_length,
            hidden_size    = hidden_size,
            num_layers     = num_layers,
            batch_first    = True,
            bidirectional  = bidirectional,
            dropout        = dropout if num_layers > 1 else 0.0,
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # 2) FC head
        rnn_out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(rnn_out_dim, out_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, input_dim=64)
        returns: (batch, out_dim)
        """
        out, _ = self.backbone(x)             # (batch, seq_len, hidden_size)
        feat   = out[:, -1, :]                # take last time step
        return self.head(feat)                # (batch, out_dim)


In [29]:
class LSTMWithHead(nn.Module):
    """
    LSTMWithHead (projected):
      • Projects raw feature vectors from `input_dim` to a compact `patch_length`
      • Feeds the projected sequence to an LSTM backbone
      • Uses the last hidden state to drive an FC head for the downstream task
    """
    def __init__(
        self,
        patch_length: int = 64,   # dimension consumed by the LSTM backbone
        hidden_size: int  = 64,   # LSTM hidden size
        num_layers: int   = 3,   # number of stacked LSTM layers
        bidirectional: bool = True,
        dropout: float      = 0.1,
        hidden_dim: int     = 256, # FC-head hidden size
        out_dim: int        = 64,  # FC-head output size
        freeze_backbone: bool = False,
    ):
        super().__init__()

        # 0) Raw 64-dim → 16-dim patch projection
        

        # 1) LSTM backbone that expects `patch_length` features
        self.backbone = nn.LSTM(
            input_size     = patch_length,
            hidden_size    = hidden_size,
            num_layers     = num_layers,
            batch_first    = True,
            bidirectional  = bidirectional,
            dropout        = dropout if num_layers > 1 else 0.0,
        )
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # 2) FC head
        lstm_out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(lstm_out_dim, out_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, input_dim=64)
        returns: (batch, out_dim)
        """
        # project raw features to patch_length
        

        # sequence modeling with LSTM
        out, _ = self.backbone(x)          # (B, seq_len, lstm_out_dim)

        # take the last time-step representation
        feat = out[:, -1, :]                    # (B, lstm_out_dim)

        # downstream head
        return self.head(feat)                  # (B, out_dim)


## fine-tuning

In [30]:
# ──────────────────────────
# Shared hyper-parameters
# ──────────────────────────
PATCH_LENGTH  = 64     # dimension fed to every backbone
HIDDEN_DIM    = 256    # head hidden dimension
D_MODEL       = 64     # internal hidden size (GRU/LSTM/Transformer)
N_LAYERS      = 12     # stacked layers
R_LAYERS      = 3      # RNN series layers -< 3
T_LAYERS      = 4      # transformer layers 12 - > 4
OUT_DIM       = 64     # head output dimension
DROPOUT       = 0.0    # dropout for recurrent / transformer blocks
MAXLEN        = 129
BIDIRECTIONAL = False   # use bidirectional RNNs
DEVICE        = "cuda"

# ──────────────────────────
# Model class catalog
# ──────────────────────────
MODEL_CATALOG = {
    # "LWM_freeze_backbone"     : LWMWithHead,
    # "LWM_pretrained_Fine_tune": LWMWithHead,
    "LWM_Fine_tune"           : LWMWithHead,
    # "GRU"                     : GRUWithHead,
    # "RNN"                     : RNNWithHead,
    # "LSTM"                    : LSTMWithHead,
    # "Transformer"             : TransformerWithHead
}

# ──────────────────────────
# Per-model constructor kwargs
# ──────────────────────────
MODEL_PARAMS = {
    # ── LWM variants ─────────────────────────────
    # "LWM_freeze_backbone": {
    #     "patch_length"    : PATCH_LENGTH,
    #     "d_model"         : D_MODEL,
    #     "max_len"         : MAXLEN,
    #     "n_layers"        : N_LAYERS,
    #     "out_dim"         : OUT_DIM,
    #     "freeze_backbone" : True,
    #     "checkpoint_path" : "./model_weights.pth",
    #     "device"          : DEVICE,
    # },
    # "LWM_pretrained_Fine_tune": {
    #     "patch_length"    : PATCH_LENGTH,
    #     "d_model"         : D_MODEL,
    #     "max_len"         : MAXLEN,
    #     "n_layers"        : N_LAYERS,
    #     "out_dim"         : OUT_DIM,
    #     "freeze_backbone" : False,
    #     "checkpoint_path" : "./model_weights.pth",
    #     "device"          : DEVICE,
    # },
    "LWM_Fine_tune": {
        "patch_length"    : PATCH_LENGTH,
        "d_model"         : D_MODEL,
        "max_len"         : MAXLEN,
        "n_layers"        : N_LAYERS,
        "out_dim"         : OUT_DIM,
        "freeze_backbone" : False,
        "checkpoint_path" : None,
        "device"          : DEVICE,
    },

    # ── GRU (projected) ──────────────────────────
    # "GRU": {
    #     "patch_length"    : PATCH_LENGTH,
    #     "d_model"         : D_MODEL,
    #     "n_layers"        : R_LAYERS,
    #     "bidirectional"   : BIDIRECTIONAL,
    #     "dropout"         : DROPOUT,
    #     "out_dim"         : OUT_DIM,
    #     "freeze_backbone" : False,
    # },
    
    # # ── Vanilla RNN (projected) ──────────────────
    # "RNN": {
    #     "patch_length"    : PATCH_LENGTH,
    #     "hidden_size"     : D_MODEL,
    #     "num_layers"      : R_LAYERS,
    #     "bidirectional"   : BIDIRECTIONAL,
    #     "dropout"         : DROPOUT,
    #     "out_dim"         : OUT_DIM,
    #     "freeze_backbone" : False,
    # },
    


    # # ── LSTM (projected) ─────────────────────────
    # "LSTM": {
    #     "hidden_size"     : D_MODEL,
    #     "num_layers"      : R_LAYERS,
    #     "bidirectional"   : BIDIRECTIONAL,
    #     "dropout"         : DROPOUT,
    #     "out_dim"         : OUT_DIM,
    #     "freeze_backbone" : False,
    # },
    

    # # ── Transformer (projected) ──────────────────
    # "Transformer": {
    #     "patch_length"    : PATCH_LENGTH,
    #     "d_model"         : D_MODEL,
    #     "n_heads"         : 8,
    #     "dim_ff"          : 256,
    #     "n_layers"        : T_LAYERS,
    #     "dropout"         : DROPOUT,
    #     "out_dim"         : OUT_DIM,
    #     "max_len"         : MAXLEN,
    #     "freeze_backbone" : False,
    # },
}


## model evaluate

In [31]:
import torch
import torch.nn.functional as F

def rmse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    Root-Mean-Squared Error
    """
    return torch.sqrt(F.mse_loss(pred, target, reduction="mean"))   # √MSE

def nmse(pred: torch.Tensor, target: torch.Tensor, eps : float = 1e-12) -> torch.Tensor:
    """
    Normalized MSE  =  E[‖ŷ − y‖²] / E[‖y‖²]
    """
    # (B, …) → (B,)  
    mse_per_sample   = ((pred - target)**2).view(pred.size(0), -1).sum(dim=1)
    power_per_sample = (target**2).view(target.size(0), -1).sum(dim=1) + eps
    return (mse_per_sample / power_per_sample).mean()



In [32]:
def masked_evaluate(model, loader, device="cuda"):
    """
    Validation loop for IterableDataset.
    Returns average RMSE and NMSE over all samples.
    """
    model.eval()
    total_rmse, total_nmse, total_samples = 0.0, 0.0, 0

    with torch.no_grad():
        for input_ids, masked_pos, target in loader:
            # Move to device
            input_ids, masked_pos, target = (
                input_ids.to(device),
                masked_pos.to(device),
                target.to(device),
            )
            # Batch size
            bs = input_ids.size(0)

            # Forward
            pred = model(input_ids, masked_pos)

            # Accumulate batch metrics
            total_rmse    += rmse(pred, target).item() * bs
            total_nmse    += nmse(pred, target).item() * bs
            total_samples += bs

    # Compute averages
    return {
        "RMSE": total_rmse / total_samples,
        "NMSE": total_nmse / total_samples
    }

In [33]:
import inspect

def unmasked_evaluate(model, loader, device, patch_length=4):
    """
    Validation loop for IterableDataset.
    Computes and returns the average RMSE and NMSE over the dataset.
    """
    model.eval()
    total_rmse, total_nmse, total_samples = 0.0, 0.0, 0

    # Inspect the model's forward signature to determine if it requires a decoder input
    sig = inspect.signature(model.forward)
    needs_tgt = len(sig.parameters) >= 3  # True if forward(self, src, tgt, ...) exists

    with torch.no_grad():
        for input_ids, target in loader:
            # Move input and target tensors to the specified device
            input_ids = input_ids.to(device)
            target = target.to(device)

            if needs_tgt:
                # Transformer models: use the last `patch_length` time steps as decoder input
                tgt = input_ids[:, -patch_length:, :]
                pred = model(input_ids, tgt)
            else:
                # Single-input models (e.g., GRU, LSTM): only the source sequence is needed
                pred = model(input_ids)

            # Accumulate weighted metrics
            batch_size = input_ids.size(0)
            total_rmse += rmse(pred, target).item() * batch_size
            total_nmse += nmse(pred, target).item() * batch_size
            total_samples += batch_size

    # Calculate average RMSE and NMSE over all samples
    avg_rmse = total_rmse / total_samples
    avg_nmse = total_nmse / total_samples

    return {
        "RMSE": avg_rmse,
        "NMSE": avg_nmse
    }


# Model Training

In [34]:
"""
Unified training / validation script
------------------------------------
* Trains every architecture listed in MODEL_CATALOG
* Chooses masked / un-masked DataLoader automatically
* Reports per-epoch speed, train/validation loss & validation scores
* Saves **best** and **last** checkpoints under ./checkpoints/
"""

# ─────────────────────────────────────────────
# 0) Globals and hyper-parameters
# ─────────────────────────────────────────────
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion   = nn.MSELoss().to(device)

NUM_EPOCHS  = 150
LR          = 1e-4                         # learning-rate
CKPT_DIR    = Path("checkpoints")          # where *.pth files will be stored
CKPT_DIR.mkdir(exist_ok=True)

total_start = time.time()                  # wall-clock timer for *all* models
results     = {}                           # best-epoch NMSE(dB) for every model

# ─────────────────────────────────────────────
# 1) Train / validate each model
# ─────────────────────────────────────────────
for model_name, ModelCls in MODEL_CATALOG.items():

    print(f"\n=== Training {model_name} ===")
    model_args = MODEL_PARAMS[model_name]
    model      = ModelCls(**model_args).to(device)

    # collect only trainable parameters
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if len(trainable_params) == 0:
        print(f"⚠️  '{model_name}' has no trainable parameters — skipping.")
        results[model_name] = float("nan")
        continue

    optimizer   = torch.optim.Adam(trainable_params, lr=LR)
    epoch_times = []                       # per-epoch training duration
    best_nmse   = float("inf")             # track the best val-NMSE

    # pick loaders / evaluation fn based on model family
    uses_mask  = model_name.startswith("LWM_")
    tr_loader  = masked_train_loader if uses_mask else unmasked_train_loader
    val_loader = masked_val_loader  if uses_mask else unmasked_val_loader
    eval_fn    = masked_evaluate    if uses_mask else unmasked_evaluate

    # ── EPOCH LOOP ──────────────────────────
    for epoch in range(1, NUM_EPOCHS + 1):

        # ---------- TRAIN ----------
        t0 = time.time()
        model.train()
        run_loss = 0.0

        pbar = tqdm(tr_loader,
                    desc=f"[{model_name} {epoch:02d}/{NUM_EPOCHS}] train",
                    leave=False)

        # input_ids.shape = (B,L,64) -> (256, 14, 64) -> (3584,64)
        

        for b, batch in enumerate(pbar, 1):
            # prepare inputs
            # xb    → shape: (batch_size, seq_len, vec_len) 
            # mpos  → shape: (batch_size, 1) 
            # yb    → shape: (batch_size, vec_len)            
            if uses_mask:
                xb, mpos, yb = [x.to(device) for x in batch]
                pred = model(xb, mpos).squeeze(-1)
            else:
                xb, yb = [x.to(device) for x in batch]
                if model_name == "Transformer":
                    tgt = xb[:,4:,:]
                    pred = model(xb, tgt)
                else:
                    pred = model(xb)

            # forward/backward
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            run_loss += loss.item()
            if b % 100 == 0:
                pbar.set_postfix(train_loss=run_loss / b)

        epoch_times.append(time.time() - t0)
        avg_train_loss = run_loss / b

        # ---------- VALID ----------
        model.eval()
        val_run_loss = 0.0
        with torch.no_grad():
            for b_val, batch_val in enumerate(val_loader, 1):
                if uses_mask:
                    xb_val, mpos_val, yb_val = [x.to(device) for x in batch_val]
                    pred_val = model(xb_val, mpos_val).squeeze(-1)
                else:
                    xb_val, yb_val = [x.to(device) for x in batch_val]
                    if model_name == "Transformer":
                        tgt_val = xb_val[:,4:,:]
                        pred_val = model(xb_val, tgt_val)
                    else:
                        pred_val = model(xb_val)

                loss_val = criterion(pred_val, yb_val)
                val_run_loss += loss_val.item()

        val_avg_loss = val_run_loss / b_val

        # compute other validation metrics
        metrics      = eval_fn(model, val_loader, device)
        val_rmse     = metrics["RMSE"]
        val_nmse     = metrics["NMSE"]
        val_nmse_db  = 10 * torch.log10(torch.tensor(val_nmse)).item()

        # save best checkpoint
        if val_nmse < best_nmse:
            best_nmse = val_nmse
            torch.save(
                model.state_dict(),
                CKPT_DIR / f"{model_name}_best.pth"
            )

        # print epoch summary (including validation loss)
        print(
            f"[{epoch:02d}/{NUM_EPOCHS}] "
            f"TrainLoss: {avg_train_loss:.4f}  "
            f"ValLoss: {val_avg_loss:.4f}  "
            f"Val RMSE: {val_rmse:.4f}  "
            f"Val NMSE: {val_nmse:.4e}  "
            f"Val NMSE_dB: {val_nmse_db:.1f} dB  "
            f"TrainTime: {epoch_times[-1]:.2f}s"
        )

    # after all epochs – save *last* weights
    torch.save(
        model.state_dict(),
        CKPT_DIR / f"{model_name}_last.pth"
    )

    avg_ep_time = sum(epoch_times) / len(epoch_times)
    print(f"🕒 {model_name} – avg train time / epoch: {avg_ep_time:.2f}s")

    # store best NMSE_dB for the summary
    results[model_name] = 10 * math.log10(best_nmse)

# ─────────────────────────────────────────────
# 2) Summary
# ─────────────────────────────────────────────
print("\n=== Summary of best NMSE(dB) by model ===")
for name, nmse_db in results.items():
    print(f"{name:25s}: {nmse_db if not math.isnan(nmse_db) else 'skipped':>6}")

print(f"\nTotal training time for all models: {time.time() - total_start:.2f}s")



=== Training LWM_Fine_tune ===


[01/150] TrainLoss: 0.0144  ValLoss: 0.0071  Val RMSE: 0.0797  Val NMSE: 2.6412e-02  Val NMSE_dB: -15.8 dB  TrainTime: 246.33s


[02/150] TrainLoss: 0.0043  ValLoss: 0.0056  Val RMSE: 0.0724  Val NMSE: 2.1159e-02  Val NMSE_dB: -16.7 dB  TrainTime: 274.01s


[03/150] TrainLoss: 0.0027  ValLoss: 0.0048  Val RMSE: 0.0665  Val NMSE: 1.7907e-02  Val NMSE_dB: -17.5 dB  TrainTime: 267.01s


[04/150] TrainLoss: 0.0021  ValLoss: 0.0045  Val RMSE: 0.0651  Val NMSE: 1.7022e-02  Val NMSE_dB: -17.7 dB  TrainTime: 261.43s


...

[146/150] TrainLoss: 0.0005  ValLoss: 0.0043  Val RMSE: 0.0633  Val NMSE: 1.6173e-02  Val NMSE_dB: -17.9 dB  TrainTime: 262.58s


[147/150] TrainLoss: 0.0005  ValLoss: 0.0043  Val RMSE: 0.0633  Val NMSE: 1.6127e-02  Val NMSE_dB: -17.9 dB  TrainTime: 262.02s


[148/150] TrainLoss: 0.0005  ValLoss: 0.0043  Val RMSE: 0.0633  Val NMSE: 1.6129e-02  Val NMSE_dB: -17.9 dB  TrainTime: 261.60s


[149/150] TrainLoss: 0.0005  ValLoss: 0.0043  Val RMSE: 0.0631  Val NMSE: 1.6018e-02  Val NMSE_dB: -18.0 dB  TrainTime: 254.04s


[150/150] TrainLoss: 0.0005  ValLoss: 0.0043  Val RMSE: 0.0631  Val NMSE: 1.6029e-02  Val NMSE_dB: -18.0 dB  TrainTime: 261.71s
🕒 LWM_Fine_tune – avg train time / epoch: 250.71s

=== Summary of best NMSE(dB) by model ===
LWM_Fine_tune            : -18.239369760809886

Total training time for all models: 51765.79s


### train_size =1454

In [35]:
dataset[0][0]['user']['channel'][3]

array([[[-7.7230516e-06-3.7325199e-06j, -7.9651563e-06-3.0329672e-06j,
         -8.1445578e-06-2.3225596e-06j, ...,
          4.0902052e-07-9.2439504e-06j, -4.2611231e-07-9.2992277e-06j,
         -1.2701220e-06-9.2776154e-06j],
        [-8.9658297e-06-5.0922256e-07j, -8.9196074e-06+2.7402834e-07j,
         -8.8062361e-06+1.0425907e-06j, ...,
         -3.1661684e-06-8.7219523e-06j, -3.9539450e-06-8.4581316e-06j,
         -4.7217295e-06-8.1212092e-06j],
        [-8.8458492e-06+3.0791930e-06j, -8.4874573e-06+3.8251083e-06j,
         -8.0681375e-06+4.5295747e-06j, ...,
         -6.2886138e-06-6.8233676e-06j, -6.9076514e-06-6.2828899e-06j,
         -7.4816817e-06-5.6838990e-06j],
        ...,
        [ 1.7968513e-06-4.9266268e-06j,  1.4072244e-06-5.0916565e-06j,
          1.0005344e-06-5.2284004e-06j, ...,
          6.3669818e-06-3.4926693e-06j,  5.9878876e-06-4.0733821e-06j,
          5.5549508e-06-4.6118416e-06j],
        [-1.2760374e-07-4.8192383e-06j, -5.0755955e-07-4.8266352e-06j,
    

## inference

In [36]:
# ─────────────────────────────────────────────
# 0)  Load the *best* checkpoints into `trained_models`
# ─────────────────────────────────────────────
CKPT_DIR = Path("checkpoints")                 # folder with *.pth files
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

trained_models = {}
for name, ModelCls in MODEL_CATALOG.items():
    ckpt_path = CKPT_DIR / f"{name}_best.pth"
    if ckpt_path.exists():
        model = ModelCls(**MODEL_PARAMS[name])     # init on CPU
        model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
        trained_models[name] = model               # keep on CPU for now
    else:
        print(f"⚠️  {ckpt_path} not found — skipping this model.")

# ─────────────────────────────────────────────
# 1)  Pure-inference timing loop (no loss / labels)
# ─────────────────────────────────────────────
torch.backends.cudnn.benchmark = True           # let cuDNN pick fastest kernels
INFER_TIME = {}                                 # {model: (total, per_batch, per_sample)}

for name, model in trained_models.items():
    uses_mask      = name.startswith("LWM_")
    is_transformer = name.startswith("Transformer")   # covers Transformer & TransformerWithHead
    v_loader       = masked_val_loader if uses_mask else unmasked_val_loader

    model = model.to(device).eval()

    # ― Warm-up (one batch) to ramp GPU clocks and cache kernels
    with torch.no_grad():
        batch = next(iter(v_loader))
        if uses_mask:
            seq, mpos, _ = [x.to(device) for x in batch]
            _ = model(seq, mpos)
        elif is_transformer:
            seq, _ = [x.to(device) for x in batch]
            tgt    = seq[:, 4:, :]                 # same slice used during training
            _ = model(seq, tgt)
        else:
            seq, _ = [x.to(device) for x in batch]
            _ = model(seq)

    # ― Timed inference pass over the entire loader
    torch.cuda.synchronize()
    t0        = time.time()
    n_batches = 0
    n_samples = 0

    with torch.no_grad():
        for batch in v_loader:
            if uses_mask:
                seq, mpos, _ = [x.to(device) for x in batch]
                _  = model(seq, mpos)
                bs = seq.size(0)
            elif is_transformer:
                seq, _ = [x.to(device) for x in batch]
                tgt    = seq[:, 4:, :]
                _  = model(seq, tgt)
                bs = seq.size(0)
            else:
                seq, _ = [x.to(device) for x in batch]
                _  = model(seq)
                bs = seq.size(0)

            n_batches += 1
            n_samples += bs

    torch.cuda.synchronize()
    elapsed = time.time() - t0

    INFER_TIME[name] = (
        elapsed,                # total seconds
        elapsed / n_batches,    # seconds per batch
        elapsed / n_samples     # seconds per sample
    )

    print(f"⏱ {name:25s} | total {elapsed:6.2f}s  "
          f"| /batch {elapsed/n_batches*1e3:6.2f} ms  "
          f"| /sample {elapsed/n_samples*1e3:6.2f} ms")

# ─────────────────────────────────────────────
# 2)  Pretty summary table
# ─────────────────────────────────────────────
print("\n=== Inference-time summary ===")
header = f"{'model':25s} | {'total [s]':>9} | {'/batch [ms]':>12} | {'/sample [ms]':>13}"
print(header)
print("-" * len(header))
for n, (tot, pb, ps) in INFER_TIME.items():
    print(f"{n:25s} | {tot:9.4f} | {pb*1e3:12.4f} | {ps*1e3:13.4f}")


⏱ LWM_Fine_tune             | total  41.10s  | /batch  83.19 ms  | /sample   0.33 ms

=== Inference-time summary ===
model                     | total [s] |  /batch [ms] |  /sample [ms]
--------------------------------------------------------------------
LWM_Fine_tune             |   41.0983 |      83.1949 |        0.3251


# Compare trainable parameters
## define trainable parameters and total parameters

In [37]:
def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def count_total_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


In [38]:
# ─────────────────────────────────────────────
# Report trainable parameters for every model
# ─────────────────────────────────────────────
print("\n=== Trainable parameters per model ===")
for name, ModelCls in MODEL_CATALOG.items():
    # instantiate model with its params (on CPU is fine for counting)
    model = ModelCls(**MODEL_PARAMS[name])
    count = count_trainable_params(model)
    print(f"{name:25s}: {count:,}")



=== Trainable parameters per model ===
LWM_Fine_tune            : 614,064


In [39]:
# ─────────────────────────────────────────────
# Report total parameters for every model
# ─────────────────────────────────────────────
print("\n===  Total parameters per model ===")
for name, ModelCls in MODEL_CATALOG.items():
    # instantiate model with its params (on CPU is fine for counting)
    model = ModelCls(**MODEL_PARAMS[name])
    count = count_total_params(model)
    print(f"{name:25s}: {count:,}")



===  Total parameters per model ===
LWM_Fine_tune            : 614,064


# Total Time

In [40]:
end = time.time()

elapsed = end - start                                
h, rem = divmod(elapsed, 3600)                       
m, s  = divmod(rem, 60)

print(f"Total elapsed time: {elapsed:.2f} seconds "
      f"({int(h)} h {int(m)} m {s:.2f} s)")

Total elapsed time: 52131.72 seconds (14 h 28 m 51.72 s)
